In [2]:
# Bibliotecas para trabalhar com os dados
import numpy as np
import pandas as pd
import pingouin as pg

# Seleção e avaliação dos modelos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

# Preparação dos dados
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Modelos preditivos
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gráficos
import plotly.express as px
import plotly.io as pio

#sistema
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

pio.templates.default = 'plotly_dark'
px.defaults.height = 500
px.defaults.color_continuous_scale = px.colors.sequential.Viridis

### Leitura das transações
Preservamos os IDs como texto, convertemos `t_dat` para data e arredondamos os preços para quatro casas. `N_LINHAS` seleciona as primeiras linhas do arquivo, não uma amostra aleatória. Execute o notebook a partir desta pasta, pois as leituras de clientes e artigos abaixo usam caminhos relativos.

In [3]:
PASTA_DADOS = Path(".")
if not (PASTA_DADOS / "transactions_train.csv").is_file():
    PASTA_DADOS = Path("H&M - Hackaton")

N_LINHAS = 1_000_000  # None para ler o arquivo completo.

df_transaction = pd.read_csv(
    PASTA_DADOS / "transactions_train.csv",
    index_col=False,
    dtype={"customer_id": "string", "article_id": "string"},
    nrows=N_LINHAS,
)

df_transaction['price'] = df_transaction['price'].astype(float).round(4)

df_transaction['t_dat'] = pd.to_datetime(df_transaction['t_dat'], format='%Y-%m-%d')

df_transaction

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2
...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2


### Cadastro de clientes e primeiro merge
O `left` mantém as transações e `validate='many_to_one'` impede duplicações causadas por IDs repetidos no cadastro. Atenção: limitar clientes e transações ao mesmo número de linhas não seleciona as mesmas pessoas. Mais abaixo recuperamos o cadastro completo em uma cópia para exploração.

In [4]:

df_clientes = pd.read_csv(
    "customers.csv",
    index_col=False,
    dtype={"customer_id": "string"},
    nrows = N_LINHAS
)

display(df_clientes)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
...,...,...,...,...,...,...,...
999998,ba9ade47e568d76b62f274b023fbc73acfba04e38c1082...,1.0,1.0,ACTIVE,Regularly,62.0,0c9f3dc39438e945c5c8fbafbbba0d67b3fd627f667084...
999999,ba9ae8434966611eb42e03e0bfca7eac3e7e1c16dbc210...,NaN,NaN,ACTIVE,NONE,32.0,caaebcb2559fb0b6148b762b2341fc9064936aa1baf4bc...


In [5]:
df_main = df_transaction.merge(
      df_clientes[['customer_id', 'club_member_status']]
    , on='customer_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE
...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE


### Cadastro de artigos e segundo merge
Os textos foram padronizados em minúsculas, sem espaços nas pontas e com `_` entre palavras. O `.dropna()` dentro da atribuição não exclui linhas de `df_articles`: o pandas realinha pelo índice e mantém os valores ausentes. A descrição foi removida e o grupo do produto foi acrescentado ao `df_main`.

In [6]:
df_articles = pd.read_csv(
    "articles.csv",
    index_col=False,
    dtype={"article_id": "string"},
    nrows=N_LINHAS
)

df_articles.drop(columns=['detail_desc'], inplace=True)

for column in ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_name', 'index_group_name', 'section_name', 'garment_group_name']:
    df_articles[column] = (
            df_articles[column]
            .astype("string")
            .dropna()
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", "_", regex=True)
    )

df_articles

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,0108775015,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,9,black,4,dark,5,black,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
1,0108775044,108775,strap_top,253,vest_top,garment_upper_body,1010016,solid,10,white,3,light,9,white,1676,jersey_basic,A,ladieswear,1,ladieswear,16,womens_everyday_basics,1002,jersey_basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105540,0957375001,957375,claire_hair_claw,72,hair_clip,accessories,1010016,solid,9,black,4,dark,5,black,3946,small_accessories,D,divided,2,divided,52,divided_accessories,1019,accessories
105541,0959461001,959461,lounge_dress,265,dress,garment_full_body,1010016,solid,11,off_white,1,dusty_light,9,white,1641,jersey,A,ladieswear,1,ladieswear,18,womens_trend,1005,jersey_fancy


In [7]:
df_main = df_main.merge(
      df_articles[['article_id', 'product_group_name']]
    , on='article_id'
    , how='left'
    , validate='many_to_one'
)

df_main

,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,product_group_name
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.0508,2,ACTIVE,underwear
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.0305,2,ACTIVE,underwear
...,...,...,...,...,...,...,...
999998,2018-10-11,46193b292b8c4cbef8d2b75a75b92584c4fef588b07171...,0524825019,0.0339,2,ACTIVE,garment_lower_body
999999,2018-10-11,461d77eece32034c78a9c1425f84b707bffed6d86368c0...,0461414008,0.0393,2,ACTIVE,shoes


### Conferências e gráficos iniciais
`describe()` e `info()` mostram distribuição, tipos e ausências. Nos gráficos, a contagem representa **linhas de itens vendidos**, não pedidos distintos: não temos um identificador de pedido. O último dia pode estar incompleto por causa de `N_LINHAS`.

In [8]:
with pd.option_context("display.max_rows", None):
    display(df_main.describe())
    df_main.info()


,t_dat,price,sales_channel_id
count,1000000,1000000.000000,1000000.000000
mean,2018-09-29 21:07:43.190400256,0.030177,1.662943
min,2018-09-20 00:00:00,0.000700,1.000000
25%,2018-09-25 00:00:00,0.016900,1.000000
50%,2018-09-29 00:00:00,0.025400,2.000000
75%,2018-10-05 00:00:00,0.033900,2.000000
max,2018-10-11 00:00:00,0.591500,2.000000
std,NaN,0.022915,0.472705


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   t_dat               1000000 non-null  datetime64[ns]
 1   customer_id         1000000 non-null  string        
 2   article_id          1000000 non-null  string        
 3   price               1000000 non-null  float64       
 4   sales_channel_id    1000000 non-null  int64         
 5   club_member_status  729572 non-null   object        
 6   product_group_name  1000000 non-null  string        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1), string(3)
memory usage: 53.4+ MB


In [9]:
import plotly.express as px

resumo = (
    df_main["product_group_name"]
    .value_counts()
    .reset_index()
)

fig = px.bar(
    resumo,
    x="product_group_name",
    y="count",
    title="Quantidade de artigos vendidos por categoria",
    labels={
        "product_group_name": "Categoria",
        "count": "Quantidade vendida"
    }
)

fig.show()


compras_dia = (
    df_main.groupby("t_dat")
    .size()
    .reset_index(name="qtd_compras")
)

fig = px.line(
    compras_dia,
    x="t_dat",
    y="qtd_compras",
    title="Evolução diária das compras"
)

fig.show()

### Estatísticas de preço
Correção pontual: `.agg()` recebe as funções em uma **lista**. Mantivemos as mesmas cinco estatísticas; antes, `'mean'` era interpretado como argumento de eixo e gerava erro.

In [9]:
estatisticas_preco = df_main['price'].agg([
      'count'
    , 'mean'
    , 'median'
    , 'std'
    , 'skew'
])

display(estatisticas_preco)

count     1000000.000000
mean            0.030177
median          0.025400
std             0.022915
skew            3.645124
Name: price, dtype: float64

### Correlações exploratórias — preparação
Vamos trabalhar em `df_exploracao`, mantendo o `df_main` como foi montado. Relemos apenas ID e status de **todos** os clientes para corrigir as ausências geradas pelo limite do cadastro. Status realmente ausentes continuam ausentes, sem inventar uma categoria.

As correlações abaixo usam preço e canal; status e grupo de produto ficam disponíveis para futuras comparações por categoria.

In [ ]:
df_cadastro = pd.read_csv(
    PASTA_DADOS / 'customers.csv',
    usecols=['customer_id', 'club_member_status'],
    dtype={'customer_id': 'string'}
)

df_exploracao = df_main.drop(columns=['club_member_status']).merge(
      df_cadastro
    , on='customer_id'
    , how='left'
    , validate='many_to_one'
    , indicator='cadastro'
)

print('Linhas antes e depois:',                 len(df_main), len(df_exploracao))
print('Status ausentes no df_main:',            df_main['club_member_status'].isna().sum())
print('Status ausentes após reler o cadastro:', df_exploracao['club_member_status'].isna().sum())
print('Linhas sem cliente no cadastro:',        df_exploracao['cadastro'].eq('left_only').sum())
print('Linhas sem grupo de produto:',           df_exploracao['product_group_name'].isna().sum())

Linhas antes e depois: 1000000 1000000
Status ausentes no df_main: 270428
Status ausentes após reler o cadastro: 10235
Linhas sem cliente no cadastro: 0
Linhas sem grupo de produto: 0


### O que entra na correlação
**Pearson** mede relação linear; **Spearman** mede se as variáveis tendem a crescer ou diminuir juntas pela ordem dos valores. Ambos variam de -1 a 1. Valores próximos de zero não descartam outros tipos de relação.

IDs, datas e nomes de categorias não entram como números. O canal vira `canal_2`: **1 = canal 2; 0 = canal 1**. Com dois canais, Pearson nessa variável binária mede a associação entre preço e grupo (correlação ponto-bisserial); não significa que um canal seja “maior” que o outro. Não atribuímos nomes aos canais sem confirmar o dicionário dos dados.

Usamos as mesmas linhas válidas nas duas análises: cliente identificado, preço finito e canal 1 ou 2. Informamos qualquer exclusão. Mantemos os preços arredondados que você preparou; isso pode criar empates, inclusive no Spearman. `price` permanece na escala do arquivo, sem atribuir moeda.

In [11]:
if not df_exploracao['sales_channel_id'].dropna().isin([1, 2]).all():
    raise ValueError('Há outros canais na base. Revise a comparação entre os canais 1 e 2.')

df_analise = df_exploracao.loc[
      df_exploracao['customer_id'].notna()
    & np.isfinite(df_exploracao['price'])
    & df_exploracao['sales_channel_id'].isin([1, 2]),
    ['customer_id', 'price', 'sales_channel_id']
].copy()

df_analise['canal_2'] = df_analise['sales_channel_id'].eq(2).astype(int)

periodo = 'Período sem datas válidas'
if df_exploracao['t_dat'].notna().any():
    periodo = f"{df_exploracao['t_dat'].min():%d/%m/%Y} a {df_exploracao['t_dat'].max():%d/%m/%Y}"
print('Período do recorte:', periodo)
print('Linhas usadas:', len(df_analise))
print('Linhas excluídas:', len(df_exploracao) - len(df_analise))

resumo_canal = df_analise.groupby('sales_channel_id').agg(
      qtd_itens=('price', 'size')
    , qtd_clientes=('customer_id', 'nunique')
    , preco_medio=('price', 'mean')
    , preco_mediano=('price', 'median')
    , desvio_preco=('price', 'std')
)

display(resumo_canal.round(4))

Período do recorte: 20/09/2018 a 11/10/2018
Linhas usadas: 1000000
Linhas excluídas: 0


,qtd_itens,qtd_clientes,preco_medio,preco_mediano,desvio_preco
sales_channel_id,,,,,
1,337057,107336,0.0265,0.0244,0.0178
2,662943,124927,0.0321,0.0254,0.0249


### Pearson e Spearman por item vendido
Cada linha tem o mesmo peso; clientes com mais itens aparecem mais vezes. Sinal positivo entre `price` e `canal_2` indica associação com preços maiores no canal 2; sinal negativo, com preços menores. Isso pode refletir diferenças nos produtos vendidos, não um efeito do canal.

A escala de cores é fixa de -1 a 1 nas quatro matrizes. Células vazias/`NaN` indicam correlação indefinida (por exemplo, coluna constante ou menos de três observações), não correlação zero. O mínimo de três apenas evita pares insuficientes; não garante evidência estatística.

In [12]:
corr_pearson = df_analise[['price', 'canal_2']].corr(method='pearson', min_periods=3)
corr_spearman = df_analise[['price', 'canal_2']].corr(method='spearman', min_periods=3)

for metodo, matriz in [('Pearson', corr_pearson), ('Spearman', corr_spearman)]:
    fig = px.imshow(
        matriz,
        text_auto='.2f',
        zmin=-1,
        zmax=1,
        color_continuous_scale=['#3274A1', '#F3F3F3', '#C97932'],
        labels={'color': 'Correlação'},
        title=f'{metodo} — preço e canal por item<br><sup>{periodo} | {len(df_analise):,} itens</sup>'
    )
    fig.update_layout(height=500, margin=dict(t=90, b=70))
    fig.show()

### Perfil observado por cliente
Agora cada cliente ocupa uma linha e tem o mesmo peso nas matrizes. Calculamos quantidade de itens, preço médio, mediana, desvio dos preços e proporção de itens no canal 2.

`proporcao_canal_2 = 0` significa só canal 1 no recorte; `1`, só canal 2; `0.5`, metade dos itens em cada canal. A mediana resume o preço habitual **observado** e o desvio mostra sua dispersão. Não são medidas de fidelidade futura nem de preço por pedido.

Para comparar a dispersão, usamos clientes com pelo menos **dois itens**. Isso não exige duas visitas: os itens podem ser do mesmo dia. Clientes com apenas um item ficam em `perfil_clientes`, mas fora dessas matrizes. Média e mediana descrevem o mesmo preço, então uma correlação alta entre elas é esperada.

In [13]:
perfil_clientes = df_analise.groupby('customer_id').agg(
      qtd_itens=('price', 'size')
    , preco_medio=('price', 'mean')
    , preco_mediano=('price', 'median')
    , desvio_preco=('price', 'std')
    , proporcao_canal_2=('canal_2', 'mean')
).reset_index()

MIN_ITENS = 2
perfil_correlacao = perfil_clientes.loc[perfil_clientes['qtd_itens'] >= MIN_ITENS].copy()

print('Clientes no recorte válido:', len(perfil_clientes))
print('Clientes nas matrizes:', len(perfil_correlacao))
print('Clientes abaixo do mínimo de itens:', len(perfil_clientes) - len(perfil_correlacao))

display(perfil_clientes.head())

Clientes no recorte válido: 214936
Clientes nas matrizes: 167591
Clientes abaixo do mínimo de itens: 47345


,customer_id,qtd_itens,preco_medio,preco_mediano,desvio_preco,proporcao_canal_2
0,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,8,0.036425,0.02965,0.020234,0.875
1,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,3,0.044033,0.05080,0.011720,1.000
2,00007d2de826758b65a93dd24ce629ed66842531df6699...,23,0.015961,0.01520,0.007881,1.000
3,00009c2aeae8761f738e4f937d9be6b49861a66339c2b1...,1,0.025400,0.02540,NaN,0.000
4,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,6,0.029367,0.02710,0.003512,1.000


In [14]:
colunas_correlacao = [
      'qtd_itens'
    , 'preco_medio'
    , 'preco_mediano'
    , 'desvio_preco'
    , 'proporcao_canal_2'
]

corr_clientes_pearson = perfil_correlacao[colunas_correlacao].corr(method='pearson', min_periods=3)
corr_clientes_spearman = perfil_correlacao[colunas_correlacao].corr(method='spearman', min_periods=3)

rotulos = ['Qtd. itens', 'Preço médio', 'Preço mediano', 'Desvio do preço', 'Proporção canal 2']

for metodo, matriz in [('Pearson', corr_clientes_pearson), ('Spearman', corr_clientes_spearman)]:
    fig = px.imshow(
        matriz,
        x=rotulos,
        y=rotulos,
        text_auto='.2f',
        zmin=-1,
        zmax=1,
        color_continuous_scale=['#3274A1', '#F3F3F3', '#C97932'],
        labels={'color': 'Correlação'},
        title=f'{metodo} — perfil dos clientes<br><sup>{periodo} | {len(perfil_correlacao):,} clientes com pelo menos {MIN_ITENS} itens</sup>'
    )
    fig.update_layout(height=600, margin=dict(t=90, b=110))
    fig.update_xaxes(tickangle=-25)
    fig.show()

### Hipóteses para investigar depois
- Quem concentra itens no canal 2 apresenta preço mediano diferente? Compare `preco_mediano` com `proporcao_canal_2` nos dois métodos.
- Clientes com mais itens apresentam maior ou menor dispersão de preços? Compare `qtd_itens` com `desvio_preco`; o número de observações também influencia a estimativa da dispersão.
- A relação entre preço e canal continua ao comparar o mesmo grupo de produto? Essa comparação ajuda a separar associação com canal de diferenças no mix de produtos.

Essas matrizes levantam hipóteses; não testam causalidade, significância ou permanência de preferência. O recorte inicial é curto, termina possivelmente no meio do dia e contém itens repetidos por cliente. Correlação por item e por cliente responde a perguntas diferentes. Antes de modelar, será necessário definir o alvo e separar passado e futuro: um resumo calculado com todo o período pode vazar informação futura.

Referências de código: [pandas — correlações](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) e [Plotly — heatmaps anotados](https://plotly.com/python/annotated-heatmap/).